#

In [2]:
# imports
import pandas as pd
import os

# Helpers
from Scripts.q1k_preprocessing import create_diagnosis_columns, create_IQ_column
from Scripts.q1k_stats import get_group_stats

In [3]:
root_dir = '/Users/emmanuelle.coutu-nadeau/Code/NED LAB/GENiAL'
database = pd.read_csv(os.path.join(root_dir, 'Data/Q1KDatabase-ECNEEGIQGENCHUSJ_DATA_flattened_cleaned_cnv.csv'))

# Table of Contents
1. [Create new diagnosis columns](#Create-new-diagnosis-columns)
2. [Create group column](#create-group-columns)
3. [Sample Overview](#sample-overview)

# Create new diagnosis columns

In [5]:
output_path = os.path.join(root_dir, 'Data/Q1KDatabase-ECNEEGIQGENCHUSJ_DATA_flattened_cleaned_cnv_renamedcols.csv')

# Apply the function to create new diagnosis columns
create_diagnosis_columns(database, output_path)

database_clean_cols = pd.read_csv(output_path)

# Display the counts for each new diagnosis column
diagnosis_columns = [
    'ASD', 'ASD_behavior', 'ADHD', 'ID', 'OCD', 'motor_disorder',
    'anxiety', 'neurological_conditions', 'genetic_disorder', 'other'
]

print("\nDiagnosis Distribution:")
for col in diagnosis_columns:
    count = database_clean_cols[col].sum(skipna=True)  # Skip NaN values in sum
    percentage = (count / len(database) * 100).round(1)
    missing_count = database_clean_cols[col].isna().sum()
    missing_percentage = (missing_count / len(database) * 100).round(1)
    print(f"{col}: {count} ({percentage}%) - Missing / Blank: {missing_count} ({missing_percentage}%)")



Database with new columns saved to: /Users/emmanuelle.coutu-nadeau/Code/NED LAB/GENiAL/Data/Q1KDatabase-ECNEEGIQGENCHUSJ_DATA_flattened_cleaned_cnv_renamedcols.csv

Diagnosis Distribution:
ASD: 58.0 (22.4%) - Missing / Blank: 116 (44.8%)
ASD_behavior: 25.0 (9.7%) - Missing / Blank: 118 (45.6%)
ADHD: 84.0 (32.4%) - Missing / Blank: 112 (43.2%)
ID: 31.0 (12.0%) - Missing / Blank: 119 (45.9%)
OCD: 0.0 (0.0%) - Missing / Blank: 237 (91.5%)
motor_disorder: 6.0 (2.3%) - Missing / Blank: 231 (89.2%)
anxiety: 36.0 (13.9%) - Missing / Blank: 109 (42.1%)
neurological_conditions: 44.0 (17.0%) - Missing / Blank: 215 (83.0%)
genetic_disorder: 8.0 (3.1%) - Missing / Blank: 251 (96.9%)
other: 108 (41.7%) - Missing / Blank: 0 (0.0%)


In [6]:
output_path = os.path.join(root_dir, 'Data/Q1KDatabase-ECNEEGIQGENCHUSJ_DATA_flattened_cleaned_cnv_renamedcols_IQ.csv')

# Apply the function to create new diagnosis columns
create_IQ_column(database_clean_cols, output_path)

database_clean_cols_IQ = pd.read_csv(output_path)

Database with IQ column saved to: /Users/emmanuelle.coutu-nadeau/Code/NED LAB/GENiAL/Data/Q1KDatabase-ECNEEGIQGENCHUSJ_DATA_flattened_cleaned_cnv_renamedcols_IQ.csv


# Create Group columns

In [7]:
database_clean_cols_IQ_groups = database_clean_cols_IQ.copy()

# Convert diagnosis columns to numeric, coercing errors to NaN
diagnosis_cols = ['Control', 'Neurodev', 'Genetic_carrier', 'Unknown', 'Other_non-neurodev']
for col in diagnosis_cols:
    database_clean_cols_IQ_groups[col] = pd.to_numeric(database_clean_cols_IQ_groups[col], errors='coerce').fillna(0)

# Create a new Group column, default to empty string
database_clean_cols_IQ_groups['Group'] = ''

# Assign "Control" if Control == 1
control_mask = (database_clean_cols_IQ_groups['Control'] == 1)
database_clean_cols_IQ_groups.loc[control_mask, 'Group'] = 'Control'

# Assign "Neurodev" if Control == 0, Neurodev == 1, Genetic_carrier == 0
neurodev_only_mask = (
    (database_clean_cols_IQ_groups['Control'] == 0) &
    (database_clean_cols_IQ_groups['Neurodev'] == 1) &
    (database_clean_cols_IQ_groups['Genetic_carrier'] == 0)
)
database_clean_cols_IQ_groups.loc[neurodev_only_mask, 'Group'] = 'Neurodev'

# Assign "Neurodev_gmc" if Control == 0, Neurodev == 1, Genetic_carrier == 1
neurodev_gmc_mask = (
    (database_clean_cols_IQ_groups['Control'] == 0) &
    (database_clean_cols_IQ_groups['Neurodev'] == 1) &
    (database_clean_cols_IQ_groups['Genetic_carrier'] == 1)
)
database_clean_cols_IQ_groups.loc[neurodev_gmc_mask, 'Group'] = 'Neurodev_gmc'

# Assign "Gmc_only" if Control == 0, Neurodev == 0, Genetic_carrier == 1
gmc_only_mask = (
    (database_clean_cols_IQ_groups['Control'] == 0) &
    (database_clean_cols_IQ_groups['Neurodev'] == 0) &
    (database_clean_cols_IQ_groups['Genetic_carrier'] == 1)
)
database_clean_cols_IQ_groups.loc[gmc_only_mask, 'Group'] = 'Gmc_only'

# Assign "unknown" if Unknown == 1 AND no other group indicators are present
unknown_mask = (
    (database_clean_cols_IQ_groups['Unknown'] == 1) &
    (database_clean_cols_IQ_groups['Control'] == 0) &
    (database_clean_cols_IQ_groups['Neurodev'] == 0) &
    (database_clean_cols_IQ_groups['Genetic_carrier'] == 0)
)
database_clean_cols_IQ_groups.loc[unknown_mask, 'Group'] = 'unknown'

# Assign "other" if other_non-neurodev == 1 AND no other group indicators are present
other_mask = (
    (database_clean_cols_IQ_groups['Unknown'] == 0) &
    (database_clean_cols_IQ_groups['Control'] == 0) &
    (database_clean_cols_IQ_groups['Neurodev'] == 0) &
    (database_clean_cols_IQ_groups['Genetic_carrier'] == 0) &
    (database_clean_cols_IQ_groups['Other_non-neurodev'] == 1)
)
database_clean_cols_IQ_groups.loc[other_mask, 'Group'] = 'Other_non-neurodev'

# Assign "NAN" if all relevant columns are 0 (Control, Neurodev, Genetic_carrier, Unknown, Other_non-neurodev)
all_zero_mask = (
    (database_clean_cols_IQ_groups['Control'] == 0) &
    (database_clean_cols_IQ_groups['Neurodev'] == 0) &
    (database_clean_cols_IQ_groups['Genetic_carrier'] == 0) &
    (database_clean_cols_IQ_groups['Unknown'] == 0) &
    (database_clean_cols_IQ_groups['Other_non-neurodev'] == 0)
)
database_clean_cols_IQ_groups.loc[all_zero_mask, 'Group'] = 'NAN'

# Save the updated dataframe
output_path = os.path.join(root_dir, 'Data/Q1KDatabase-ECNEEGIQGENCHUSJ_DATA_flattened_cleaned_cnv_renamedcols_IQ_groups.csv')
database_clean_cols_IQ_groups.to_csv(output_path, index=False)

# Display group sizes
print("\n-------------- \nGroup Sizes:")
print(f"Control: {len(database_clean_cols_IQ_groups[database_clean_cols_IQ_groups['Group'] == 'Control'])}")
print(f"Neurodev: {len(database_clean_cols_IQ_groups[database_clean_cols_IQ_groups['Group'] == 'Neurodev'])}")
print(f"Neurodev_gmc: {len(database_clean_cols_IQ_groups[database_clean_cols_IQ_groups['Group'] == 'Neurodev_gmc'])}")
print(f"Gmc_only: {len(database_clean_cols_IQ_groups[database_clean_cols_IQ_groups['Group'] == 'Gmc_only'])}")
print(f"Unknown: {len(database_clean_cols_IQ_groups[database_clean_cols_IQ_groups['Group'] == 'unknown'])}")
print(f"Other_non-neurodev: {len(database_clean_cols_IQ_groups[database_clean_cols_IQ_groups['Group'] == 'Other_non-neurodev'])}")
print(f"NAN: {len(database_clean_cols_IQ_groups[database_clean_cols_IQ_groups['Group'] == 'NAN'])}")
print(f"Total: {len(database_clean_cols_IQ_groups)}")

print("\n-------------- \nFamily members count:")
proband_count = database_clean_cols_IQ_groups[database_clean_cols_IQ_groups['participant_id'].str.contains('_P')].shape[0]
print(f"Number of probands: {proband_count}")
sibling_count = database_clean_cols_IQ_groups[database_clean_cols_IQ_groups['participant_id'].str.contains('_S')].shape[0]
print(f"Number of siblings: {sibling_count}")
father_count = database_clean_cols_IQ_groups[database_clean_cols_IQ_groups['participant_id'].str.contains('_F')].shape[0]
print(f"Number of fathers: {father_count}")
mother_count = database_clean_cols_IQ_groups[database_clean_cols_IQ_groups['participant_id'].str.contains('_M')].shape[0]
print(f"Number of mothers: {mother_count}")
other_count = database_clean_cols_IQ_groups[~database_clean_cols_IQ_groups['participant_id'].str.contains('_[PSFM]')].shape[0]
print(f"Number of others: {other_count}")
print(f"Total: {proband_count+sibling_count+father_count+mother_count+other_count}")




-------------- 
Group Sizes:
Control: 77
Neurodev: 81
Neurodev_gmc: 68
Gmc_only: 13
Unknown: 12
Other_non-neurodev: 4
NAN: 4
Total: 259

-------------- 
Family members count:
Number of probands: 99
Number of siblings: 57
Number of fathers: 44
Number of mothers: 57
Number of others: 2
Total: 259


# Add demographic info

In [8]:
from Scripts.clean_demographics import clean_demographic_data, merge_demographics_to_main

In [9]:
# Step 1: Clean the demographic data
root_dir = '/Users/emmanuelle.coutu-nadeau/Code/NED LAB/GENiAL'
input_path = os.path.join(root_dir, 'Data/Q1KDatabase-ECNDEMOG_DATA.csv')
output_path = os.path.join(root_dir, 'Data/Q1KDatabase-ECNDEMOG_DATA_cleaned.csv')

clean_demographic_data(input_path, output_path)

Cleaned demographic data saved to: /Users/emmanuelle.coutu-nadeau/Code/NED LAB/GENiAL/Data/Q1KDatabase-ECNDEMOG_DATA_cleaned.csv


In [10]:
# Step 2: Merge into your main file
root_dir = '/Users/emmanuelle.coutu-nadeau/Code/NED LAB/GENiAL'
main_path = os.path.join(root_dir, 'Data/Q1KDatabase-ECNEEGIQGENCHUSJ_DATA_flattened_cleaned_cnv_renamedcols_IQ_groups.csv')
demog_path = os.path.join(root_dir, 'Data/Q1KDatabase-ECNDEMOG_DATA_cleaned.csv')
output_path = os.path.join(root_dir, 'Data/Q1KDatabase-ECNEEGIQGENCHUSJ_DATA_flattened_cleaned_cnv_renamedcols_IQ_groups_demog.csv')
merge_demographics_to_main(
    main_path,
    demog_path,
    output_path
)

Merged demographics to main data and saved to: /Users/emmanuelle.coutu-nadeau/Code/NED LAB/GENiAL/Data/Q1KDatabase-ECNEEGIQGENCHUSJ_DATA_flattened_cleaned_cnv_renamedcols_IQ_groups_demog.csv


# Sample Overview

In [11]:
# Load the final database with demographic information
root_dir = '/Users/emmanuelle.coutu-nadeau/Code/NED LAB/GENiAL'
final_database_path = os.path.join(root_dir, 'Data/Q1KDatabase-ECNEEGIQGENCHUSJ_DATA_flattened_cleaned_cnv_renamedcols_IQ_groups_demog.csv')

# Load the final database
database_final = pd.read_csv(final_database_path)

print(f"Final database loaded with {len(database_final)} participants and {len(database_final.columns)} columns")
print(f"Columns: {list(database_final.columns)}")

# Display first few rows to verify the data
print("\nFirst few rows of the final database:")
display(database_final.head())

# Basic summary statistics
print(f"\nDataset shape: {database_final.shape}")
print(f"Number of unique participants: {database_final['participant_id'].nunique()}")
print(f"Number of unique record IDs: {database_final['record_id'].nunique()}")

# Define diagnosis columns
diagnosis_columns = [
    'ASD', 'ASD_behavior', 'ADHD', 'ID', 'OCD', 'motor_disorder',
    'anxiety', 'neurological_conditions', 'genetic_disorder', 'other'
]

# Create a table to store results
results = []

# For each diagnosis column
for diagnosis in diagnosis_columns:
    row = {'Diagnosis': diagnosis}
    
    # Get overall stats without grouping
    total = len(database_final)
    count = database_final[diagnosis].sum()
    pct = (count / total * 100).round(1) if total > 0 else 0
    row['Overall'] = f"{count} ({pct}%)"
    
    # Get stats per Group
    for group in database_final['Group'].dropna().unique():
        group_data = database_final[database_final['Group'] == group]
        group_total = len(group_data)
        group_count = group_data[diagnosis].sum()
        group_pct = (group_count / group_total * 100).round(1) if group_total > 0 else 0
        row[group] = f"{group_count} ({group_pct}%)"
    
    results.append(row)

# Convert to DataFrame and display
results_df = pd.DataFrame(results)
print("\nDiagnosis Distribution (Overall and by Group):")
display(results_df)

# Add age, sex, and IQ information overall and by group
print("\nAge, Sex, and IQ Distribution (Overall and by Group):")

demographic_results = []

# Overall stats
age_mean = database_final['eeg_age_years_testdate'].mean()
age_std = database_final['eeg_age_years_testdate'].std()
sex_counts = database_final['sex'].value_counts()
total = len(database_final)
male_pct = (sex_counts.get('M', 0) / total * 100).round(1)
female_pct = (sex_counts.get('F', 0) / total * 100).round(1)
iq_mean = database_final['IQ'].mean()
iq_std = database_final['IQ'].std()

demographic_results.append({
    'Group': f'Overall (n = {total})',
    'Age (mean ± std)': f"{age_mean:.1f} ± {age_std:.1f}",
    'Male': f"{sex_counts.get('M', 0)} ({male_pct}%)",
    'Female': f"{sex_counts.get('F', 0)} ({female_pct}%)",
    'IQ (mean ± std)': f"{iq_mean:.1f} ± {iq_std:.1f}"
})

# Stats by Group
for group in database_final['Group'].dropna().unique():
    group_data = database_final[database_final['Group'] == group]
    
    # Age statistics
    group_age_mean = group_data['eeg_age_years_testdate'].mean()
    group_age_std = group_data['eeg_age_years_testdate'].std()
    
    # Sex distribution
    group_sex_counts = group_data['sex'].value_counts()
    group_total = len(group_data)
    group_male_pct = (group_sex_counts.get('M', 0) / group_total * 100).round(1)
    group_female_pct = (group_sex_counts.get('F', 0) / group_total * 100).round(1)
    
    # IQ statistics
    group_iq_mean = group_data['IQ'].mean()
    group_iq_std = group_data['IQ'].std()
    
    demographic_results.append({
        'Group': f'{group} (n = {group_total})',
        'Age (mean ± std)': f"{group_age_mean:.1f} ± {group_age_std:.1f}",
        'Male': f"{group_sex_counts.get('M', 0)} ({group_male_pct}%)",
        'Female': f"{group_sex_counts.get('F', 0)} ({group_female_pct}%)",
        'IQ (mean ± std)': f"{group_iq_mean:.1f} ± {group_iq_std:.1f}"
    })

# Display demographic results
demographic_df = pd.DataFrame(demographic_results)
display(demographic_df)

Final database loaded with 259 participants and 135 columns
Columns: ['participant_id', 'record_id', 'eeg_birthdate_v2_v2', 'eeg_age_years_testdate', 'sex', 'Control', 'Neurodev', 'Genetic_carrier', 'Unknown', 'Other_non-neurodev', 'ghf_asd', 'ghf_adhd', 'ghf_id', 'ghf_autistic_behav', 'ghf_neuro', 'ghf_anxiety', 'ghf_cog_imp', 'ghf_li', 'ghf_ld', 'ghf_delay_fmd', 'ghf_agg_behav', 'diag_asd', 'diag_intel', 'diag_adhd', 'diag_motor', 'diag_fas', 'diag_learn', 'diag_comm', 'diag_hearing', 'diag_visual', 'diag_phys', 'diag_gene', 'diag_oth', 'diag_susp_other', 'wais_globalapt_comp', 'wisc_gai_is', 'wppsi_47_gaisco_', 'leiter3_full_iq', 'bayley_cg_gsv', 'bayley_cg_ae', 'eeg_diagnosis_unk', 'eeg_diagnosis_other', 'eeget_date_v2_v2', 'eeg_rsrio_done', 'eeg_rs_done', 'eeg_to_done', 'eeg_go_done', 'eeg_vep_done', 'eeg_aep_done', 'eeg_nsp_done', 'eeg_pl_done', 'eeg_vs_done', 'eeg_as_done', 'eeg_fsp_done', 'eeg_mmn_done', 'eeget_general_notes_v2_v2', 'cfq_ment_dd_2', 'cfq_ment_ad_2', 'cfq_ment_b

,participant_id,record_id,eeg_birthdate_v2_v2,eeg_age_years_testdate,sex,Control,Neurodev,Genetic_carrier,Unknown,Other_non-neurodev,...,anxiety,neurological_conditions,genetic_disorder,other,IQ,Group,relation_to_proband,household_income,highest_education_level,family_ethnicity
0,Q1K_HSJ_100100_P,100,2017-08-03,6.557401,F,0,1,1,0,0,...,1.0,NaN,NaN,0,92.0,Neurodev_gmc,Parent/Caregiver,"$20,000 - $39,999",High school diploma or certificate,White_Caucasian
1,Q1K_HSJ_100100_S1,101,2020-10-17,3.351312,M,0,0,0,1,0,...,0.0,1.0,NaN,1,NaN,unknown,NaN,NaN,NaN,NaN
2,Q1K_HSJ_100100_F1,103,1994-01-11,30.116977,M,0,1,0,0,0,...,0.0,NaN,NaN,1,71.0,Neurodev,NaN,NaN,NaN,NaN
3,Q1K_HSJ_100104_P,104,1988-04-18,36.206082,F,0,1,1,0,0,...,0.0,1.0,NaN,1,74.0,Neurodev_gmc,Yourself,"Less than $20,000",Some high school,White_Caucasian
4,Q1K_HSJ_100105_P,105,2019-11-30,4.394227,F,0,1,1,0,0,...,0.0,NaN,NaN,1,NaN,Neurodev_gmc,Parent/Caregiver,"$150,000 - $199,999",Bachelor's degree,"Arab, White_Caucasian"



Dataset shape: (259, 135)
Number of unique participants: 259
Number of unique record IDs: 259

Diagnosis Distribution (Overall and by Group):


,Diagnosis,Overall,Neurodev_gmc,unknown,Neurodev,Gmc_only,Control,NAN,Other_non-neurodev
0,ASD,58.0 (22.4%),28.0 (41.2%),1.0 (8.3%),27.0 (33.3%),0.0 (0.0%),0.0 (0.0%),2.0 (50.0%),0.0 (0.0%)
1,ASD_behavior,25.0 (9.7%),14.0 (20.6%),0.0 (0.0%),9.0 (11.1%),1.0 (7.7%),1.0 (1.3%),0.0 (0.0%),0.0 (0.0%)
2,ADHD,84.0 (32.4%),35.0 (51.5%),3.0 (25.0%),40.0 (49.4%),5.0 (38.5%),0.0 (0.0%),1.0 (25.0%),0.0 (0.0%)
3,ID,31.0 (12.0%),26.0 (38.2%),0.0 (0.0%),5.0 (6.2%),0.0 (0.0%),0.0 (0.0%),0.0 (0.0%),0.0 (0.0%)
4,OCD,0.0 (0.0%),0.0 (0.0%),0.0 (0.0%),0.0 (0.0%),0.0 (0.0%),0.0 (0.0%),0.0 (0.0%),0.0 (0.0%)
5,motor_disorder,6.0 (2.3%),2.0 (2.9%),0.0 (0.0%),4.0 (4.9%),0.0 (0.0%),0.0 (0.0%),0.0 (0.0%),0.0 (0.0%)
6,anxiety,36.0 (13.9%),10.0 (14.7%),2.0 (16.7%),18.0 (22.2%),2.0 (15.4%),3.0 (3.9%),0.0 (0.0%),1.0 (25.0%)
7,neurological_conditions,44.0 (17.0%),29.0 (42.6%),1.0 (8.3%),14.0 (17.3%),0.0 (0.0%),0.0 (0.0%),0.0 (0.0%),0.0 (0.0%)
8,genetic_disorder,8.0 (3.1%),7.0 (10.3%),0.0 (0.0%),0.0 (0.0%),1.0 (7.7%),0.0 (0.0%),0.0 (0.0%),0.0 (0.0%)
9,other,108 (41.7%),58 (85.3%),1 (8.3%),38 (46.9%),6 (46.2%),4 (5.2%),1 (25.0%),0 (0.0%)



Age, Sex, and IQ Distribution (Overall and by Group):


,Group,Age (mean ± std),Male,Female,IQ (mean ± std)
0,Overall (n = 259),24.1 ± 16.6,133 (51.4%),126 (48.6%),100.0 ± 21.6
1,Neurodev_gmc (n = 68),11.6 ± 7.6,38 (55.9%),30 (44.1%),78.6 ± 24.3
2,unknown (n = 12),22.3 ± 17.4,8 (66.7%),4 (33.3%),114.1 ± 8.7
3,Neurodev (n = 81),23.5 ± 15.5,43 (53.1%),38 (46.9%),103.5 ± 19.7
4,Gmc_only (n = 13),35.3 ± 13.9,8 (61.5%),5 (38.5%),109.0 ± 7.3
5,Control (n = 77),33.6 ± 16.6,32 (41.6%),45 (58.4%),107.2 ± 14.4
6,NAN (n = 4),27.4 ± 16.7,3 (75.0%),1 (25.0%),103.2 ± 21.4
7,Other_non-neurodev (n = 4),33.7 ± 15.1,1 (25.0%),3 (75.0%),104.2 ± 9.5


In [12]:
# DEMOG

print("PARTICIPANT COUNTS BY DEMOGRAPHIC CATEGORIES")
print("="*60)

# Household Income counts
print("\nHousehold Income:")
print(database_final['household_income'].value_counts().sort_index())

# Education Level counts  
print("\nHighest Education Level:")
print(database_final['highest_education_level'].value_counts().sort_index())

# Family Ethnicity counts
print("\nFamily Ethnicity:")
print(database_final['family_ethnicity'].value_counts())

# Show missing values
print("\nMissing Values:")
print(f"household_income: {database_final['household_income'].isna().sum()}")
print(f"highest_education_level: {database_final['highest_education_level'].isna().sum()}")
print(f"family_ethnicity: {database_final['family_ethnicity'].isna().sum()}")

print(f"\nTotal participants: {len(database_final)}")

PARTICIPANT COUNTS BY DEMOGRAPHIC CATEGORIES

Household Income:
household_income
$100,000 - $149,999    17
$150,000 - $199,999    10
$20,000 - $39,999      10
$200,000 - $249,999    12
$250,000 - $399,999     1
$40,000 - $59,999      14
$60,000 - $79,999       9
$80,000 - $99,999      12
>$400,000               2
Less than $20,000       6
Name: count, dtype: int64

Highest Education Level:
highest_education_level
Apprenticeship or other trades certificate or diploma            11
Bachelor's degree                                                30
College, CEGEP or other non-university certificate or diploma    21
Doctorate                                                         2
Elementary school or less                                         3
High school diploma or certificate                                7
Master's degree                                                   9
Other                                                             5
Some high school                       

# Adding behavioral scores to dataset

In [13]:
# Read the behavioral scores CSV
behavioral_scores = pd.read_csv("Data/Q1KDatabase-ECNSRSASEBASCQscores_DATA.csv")

# SRS

# 1. SRS Social Cognition T Score
behavioral_scores['SRS_social_cognition_tscore'] = behavioral_scores[
    ['srsps2rs_tscore_cog_v2', 'srs2sch_tscore_cog_v2', 'srs2adself_tscore_cog_v2']
].bfill(axis=1).iloc[:, 0]

# 2. SRS Social Communication T Score
behavioral_scores['SRS_social_communication_tscore'] = behavioral_scores[
    ['srsps2rs_tscore_com_v2', 'srs2sch_tscore_com_v2', 'srs2adself_tscore_com_v2']
].bfill(axis=1).iloc[:, 0]

# 3. SRS Restrictive & Repetitive T Score
behavioral_scores['SRS_restrictive_repetitive_tscore'] = behavioral_scores[
    ['srsps2rs_tscore_rrb_v2', 'srs2sch_tscore_rrb_v2', 'srs2adself_tscore_rrb_v2']
].bfill(axis=1).iloc[:, 0]

# ASEBA 
behavioral_scores['ASEBA_internalizing_problems_tscore'] = behavioral_scores[
    ['ix_internalizing_problemsd_ts_2', 'ix_internalizing_problemsss', 'ix_externalizing_problems']
].bfill(axis=1).iloc[:, 0]

behavioral_scores['ASEBA_externalizing_problems_tscore'] = behavioral_scores[
    ['other_problems_18_59_t_sco_ts_2', 'externalizing_tscore', 'ix_externalizing_probles']
].bfill(axis=1).iloc[:, 0]

behavioral_scores['ASEBA_aggressive_behavior_tscore'] = behavioral_scores[
    ['score_ts_2', 'viii_aggressive_behavior_6', 'vii_aggressive_probleems_1']
].bfill(axis=1).iloc[:, 0]

behavioral_scores['ASEBA_attention_problems_tscore'] = behavioral_scores[
    ['v_attention_problems_18_59_ts_2', 'vi_attention_problems_6_18', 'vi_attention_problems_1_5']
].bfill(axis=1).iloc[:, 0]

behavioral_scores['ASEBA_anxious_depressed_tscore'] = behavioral_scores[
    ['i_anxious_depressed_18_59_ts_2', 'cbcl_6_18_anx_depr_tscore_v2', 'ii_anxious_depressed_1_5_t_2']
].bfill(axis=1).iloc[:, 0]

# SCQ
behavioral_scores['SCQ_score'] = behavioral_scores[['scq_a', 'scq_b']].bfill(axis=1).iloc[:, 0]

# Remove useless columns
behavioral_scores = behavioral_scores.drop(columns=['redcap_repeat_instrument', 'redcap_repeat_instance'], errors='ignore')

# Only keep record_id and the renamed columns
columns_to_keep = [
    'record_id',
    'SRS_social_cognition_tscore',
    'SRS_social_communication_tscore',
    'SRS_restrictive_repetitive_tscore',
    'ASEBA_internalizing_problems_tscore',
    'ASEBA_externalizing_problems_tscore',
    'ASEBA_aggressive_behavior_tscore',
    'ASEBA_attention_problems_tscore',
    'ASEBA_anxious_depressed_tscore',
    'SCQ_score'
]
behavioral_scores = behavioral_scores[columns_to_keep]

# For each record_id, keep only the row with the most non-null values (i.e., the "info" row)
behavioral_scores = behavioral_scores.loc[
    behavioral_scores.notnull().sum(axis=1)
    .groupby(behavioral_scores['record_id'])
    .idxmax()
].reset_index(drop=True)



In [14]:
# Merge behavioral_scores into the main dataframe and save as a new CSV
root_dir = '/Users/emmanuelle.coutu-nadeau/Code/NED LAB/GENiAL'

# Path to the main cleaned/flattened database
output_merged_path = os.path.join(root_dir, 'Data/Q1KDatabase-ECNEEGIQGENCHUSJ_DATA_flattened_cleaned_cnv_renamedcols_IQ_groups_demog_behavior.csv')
main_df = database_final.copy()

# Merge on 'record_id', keeping all rows from main_df; missing behavioral_scores will be NaN
merged_df = pd.merge(main_df, behavioral_scores, on='record_id', how='left')

# Save to new CSV
merged_df.to_csv(output_merged_path, index=False)
print(f"Merged behavioral scores and saved to: {output_merged_path}")

Merged behavioral scores and saved to: /Users/emmanuelle.coutu-nadeau/Code/NED LAB/GENiAL/Data/Q1KDatabase-ECNEEGIQGENCHUSJ_DATA_flattened_cleaned_cnv_renamedcols_IQ_groups_demog_behavior.csv


Merge genes columns together (gene_name and genes)

In [16]:
# Append 'gene_name' to 'Genes' if 'gene_name' is not empty/NaN
def append_gene_name(row):
    if pd.notnull(row.get('gene_name')) and str(row.get('gene_name')).strip() != '':
        if pd.notnull(row.get('Genes')) and str(row.get('Genes')).strip() != '':
            return f"{row['Genes']},{row['gene_name']}"
        else:
            return str(row['gene_name'])
    else:
        return row.get('Genes')

if 'Genes' in merged_df.columns and 'gene_name' in merged_df.columns:
    merged_df['affected_genes'] = merged_df.apply(append_gene_name, axis=1)

# Save to new CSV
merged_df.to_csv(output_merged_path, index=False)
print(f"Merged behavioral scores and saved to: {output_merged_path}")

Merged behavioral scores and saved to: /Users/emmanuelle.coutu-nadeau/Code/NED LAB/GENiAL/Data/Q1KDatabase-ECNEEGIQGENCHUSJ_DATA_flattened_cleaned_cnv_renamedcols_IQ_groups_demog_behavior.csv


In [17]:
# Load the merged CSV and keep only the specified columns
final_columns_to_keep = [ # demog
                         'participant_id', 
                         'record_id', 
                         'eeg_rsrio_done', 
                         'eeg_rs_done',
                         'household_income',
                         'highest_education_level',
                         'family_ethnicity',
                         'sex',
                         'eeg_age_years_testdate',
                         # diagnosis
                         'ASD',
                         'ASD_behavior',
                         'ADHD',
                         'OCD',
                         'motor_disorder',
                         'anxiety',
                         'neurological_conditions',
                         'genetic_disorder',
                         'other',
                         # genetics
                         'affected_genes',
                         'Estimated loss of Non-Verbal Intelligence Quotient',
                         'Estimated odds ratio for autism',
                         'Estimated gain of raw score of Social Responsiveness Scale',
                         'Estimated probability of being de novo',
                         'Sum LOEUF',
                         # behavioral
                         'IQ',
                         'SRS_social_cognition_tscore',
                         'SRS_social_communication_tscore',
                         'SRS_restrictive_repetitive_tscore',
                         'ASEBA_internalizing_problems_tscore',
                         'ASEBA_externalizing_problems_tscore',
                         'ASEBA_aggressive_behavior_tscore',
                         'ASEBA_attention_problems_tscore',
                         'ASEBA_anxious_depressed_tscore',
                         'SCQ_score'
                         ]
final_csv_path = os.path.join(root_dir, 'Data/Q1KDatabase-ECNEEGIQGENCHUSJ_DATA_flattened_cleaned_cnv_renamedcols_IQ_groups_demog_behavior.csv')
final_df = pd.read_csv(final_csv_path)

final_df = final_df[final_columns_to_keep]

# Optionally, save to a new CSV (or overwrite if desired)
final_output_path = os.path.join(root_dir, 'Data/Q1KDatabase-ECNEEGIQGENCHUSJ_DATA_for_cluster_analysis.csv')
final_df.to_csv(final_output_path, index=False)
print(f"Saved file with only participant_id and record_id to: {final_output_path}")


Saved file with only participant_id and record_id to: /Users/emmanuelle.coutu-nadeau/Code/NED LAB/GENiAL/Data/Q1KDatabase-ECNEEGIQGENCHUSJ_DATA_for_cluster_analysis.csv
